In [1]:
# ========================
# Week 6 - Day 1: Incident Response
# ========================

In [2]:
import re
from datetime import datetime
from collections import Counter

# Simulate SSH auth log entries (like /var/log/auth.log)
sample_logs = """
Jul 19 10:23:01 server sshd[1234]: Failed password for root from 192.168.1.100 port 22
Jul 19 10:23:02 server sshd[1234]: Failed password for root from 192.168.1.100 port 22
Jul 19 10:23:03 server sshd[1234]: Failed password for root from 192.168.1.100 port 22
Jul 19 10:23:04 server sshd[1234]: Failed password for admin from 192.168.1.100 port 22
Jul 19 10:23:05 server sshd[1234]: Failed password for admin from 192.168.1.100 port 22
Jul 19 10:23:06 server sshd[1234]: Accepted password for sharuk from 192.168.1.200 port 22
Jul 19 10:23:07 server sshd[1234]: Failed password for root from 10.0.0.50 port 22
Jul 19 10:23:08 server sshd[1234]: Failed password for root from 10.0.0.50 port 22
Jul 19 10:23:09 server sshd[1234]: Failed password for root from 10.0.0.50 port 22
Jul 19 10:23:10 server sshd[1234]: Failed password for root from 10.0.0.50 port 22
Jul 19 10:23:11 server sshd[1234]: Accepted password for sharuk from 192.168.1.200 port 22
Jul 19 10:23:12 server sshd[1234]: Failed password for guest from 172.16.0.25 port 22
"""

def analyze_logs(logs):
    print(f"\n{'='*55}")
    print(f"🔍 LOG ANALYSIS REPORT")
    print(f"{'='*55}")
    
    lines = logs.strip().split('\n')
    
    failed_attempts = []
    successful_logins = []
    ip_counter = Counter()
    
    for line in lines:
        # Extract failed attempts
        if "Failed password" in line:
            ip_match = re.search(r'from (\d+\.\d+\.\d+\.\d+)', line)
            user_match = re.search(r'for (\w+) from', line)
            if ip_match and user_match:
                ip = ip_match.group(1)
                user = user_match.group(1)
                failed_attempts.append({"ip": ip, "user": user})
                ip_counter[ip] += 1
        
        # Extract successful logins
        if "Accepted password" in line:
            ip_match = re.search(r'from (\d+\.\d+\.\d+\.\d+)', line)
            user_match = re.search(r'for (\w+) from', line)
            if ip_match and user_match:
                successful_logins.append({
                    "ip": ip_match.group(1),
                    "user": user_match.group(1)
                })
    
    # Report findings
    print(f"\n📊 SUMMARY")
    print(f"Total log entries:     {len(lines)}")
    print(f"Failed attempts:       {len(failed_attempts)}")
    print(f"Successful logins:     {len(successful_logins)}")
    
    print(f"\n🚨 SUSPICIOUS IPs (3+ failed attempts)")
    for ip, count in ip_counter.most_common():
        if count >= 3:
            print(f"  {ip:20s} → {count} failed attempts ← BRUTE FORCE SUSPECTED")
    
    print(f"\n✅ SUCCESSFUL LOGINS")
    for login in successful_logins:
        print(f"  User: {login['user']:10s} from {login['ip']}")
    
    print(f"\n💡 RECOMMENDATIONS")
    suspicious = [ip for ip, count in ip_counter.items() if count >= 3]
    if suspicious:
        print(f"  Block these IPs immediately:")
        for ip in suspicious:
            print(f"  → sudo ufw deny from {ip}")

analyze_logs(sample_logs)


🔍 LOG ANALYSIS REPORT

📊 SUMMARY
Total log entries:     12
Failed attempts:       10
Successful logins:     2

🚨 SUSPICIOUS IPs (3+ failed attempts)
  192.168.1.100        → 5 failed attempts ← BRUTE FORCE SUSPECTED
  10.0.0.50            → 4 failed attempts ← BRUTE FORCE SUSPECTED

✅ SUCCESSFUL LOGINS
  User: sharuk     from 192.168.1.200
  User: sharuk     from 192.168.1.200

💡 RECOMMENDATIONS
  Block these IPs immediately:
  → sudo ufw deny from 192.168.1.100
  → sudo ufw deny from 10.0.0.50


In [3]:
import subprocess

def analyze_real_logs():
    print(f"\n{'='*55}")
    print(f"🔍 REAL SYSTEM LOG ANALYSIS")
    print(f"{'='*55}")
    
    # Get real SSH logs from your system
    result = subprocess.run(
        ['journalctl', '-u', 'sshd', '--no-pager', '-n', '50'],
        capture_output=True, text=True
    )
    
    logs = result.stdout
    lines = logs.strip().split('\n')
    
    failed = []
    accepted = []
    ip_counter = Counter()
    
    for line in lines:
        if "Failed password" in line:
            ip_match = re.search(r'from (\d+\.\d+\.\d+\.\d+)', line)
            user_match = re.search(r'for (\w+) from', line)
            if ip_match and user_match:
                ip = ip_match.group(1)
                failed.append({"ip": ip, "user": user_match.group(1)})
                ip_counter[ip] += 1
        
        if "Accepted" in line:
            ip_match = re.search(r'from (\d+\.\d+\.\d+\.\d+)', line)
            user_match = re.search(r'for (\w+) from', line)
            if ip_match and user_match:
                accepted.append({
                    "ip": ip_match.group(1),
                    "user": user_match.group(1)
                })
    
    print(f"Last 50 SSH log entries analyzed")
    print(f"Failed attempts:   {len(failed)}")
    print(f"Successful logins: {len(accepted)}")
    
    if ip_counter:
        print(f"\n🚨 Failed login IPs:")
        for ip, count in ip_counter.most_common():
            print(f"  {ip} → {count} attempts")
    else:
        print(f"\n✅ No failed login attempts found — system clean!")
    
    if accepted:
        print(f"\n✅ Successful logins:")
        for login in accepted:
            print(f"  {login['user']} from {login['ip']}")

analyze_real_logs()


🔍 REAL SYSTEM LOG ANALYSIS
Last 50 SSH log entries analyzed
Failed attempts:   0
Successful logins: 0

✅ No failed login attempts found — system clean!


In [4]:
import time
from collections import Counter
import re

def real_time_monitor(threshold=3, check_interval=10):
    print(f"🔴 LIVE SSH MONITOR STARTED")
    print(f"Alert threshold: {threshold} failed attempts")
    print(f"Check interval: {check_interval} seconds")
    print(f"Press Ctrl+C to stop\n")
    
    import subprocess
    seen_alerts = set()
    
    try:
        while True:
            # Get recent logs
            result = subprocess.run(
                ['journalctl', '-u', 'sshd', '--no-pager',
                 '--since', '5 minutes ago'],
                capture_output=True, text=True
            )
            
            ip_counter = Counter()
            for line in result.stdout.split('\n'):
                if "Failed password" in line:
                    ip_match = re.search(r'from (\d+\.\d+\.\d+\.\d+)', line)
                    if ip_match:
                        ip_counter[ip_match.group(1)] += 1
            
            # Check for threshold breaches
            for ip, count in ip_counter.items():
                if count >= threshold and ip not in seen_alerts:
                    print(f"🚨 ALERT! {datetime.now().strftime('%H:%M:%S')}")
                    print(f"   Brute force detected from {ip}")
                    print(f"   {count} failed attempts in last 5 minutes")
                    print(f"   Run: sudo ufw deny from {ip}\n")
                    seen_alerts.add(ip)
            
            if not ip_counter:
                print(f"✅ {datetime.now().strftime('%H:%M:%S')} — No threats detected")
            
            time.sleep(check_interval)
            
    except KeyboardInterrupt:
        print(f"\n🔴 Monitor stopped")

# Run for 30 seconds
real_time_monitor(threshold=3, check_interval=10)

🔴 LIVE SSH MONITOR STARTED
Alert threshold: 3 failed attempts
Check interval: 10 seconds
Press Ctrl+C to stop

✅ 19:55:30 — No threats detected
✅ 19:55:40 — No threats detected
✅ 19:55:50 — No threats detected
✅ 19:56:00 — No threats detected
✅ 19:56:10 — No threats detected
✅ 19:56:20 — No threats detected
✅ 19:56:30 — No threats detected
✅ 19:56:40 — No threats detected
✅ 19:56:50 — No threats detected
✅ 19:57:00 — No threats detected
✅ 19:57:10 — No threats detected
✅ 19:57:20 — No threats detected
✅ 19:57:30 — No threats detected
✅ 19:57:40 — No threats detected

🔴 Monitor stopped
